In [1]:
import pandas as pd 
import statsmodels.api as sm 
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
import src.regression as kit


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [4]:
df = pd.read_csv(PROCESSED_DIR/"etf_ff5_merged.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.head()

,date,ticker,return,mkt_rf,smb,hml,rmw,cma,rf,excess_return
0,2015-02-28,IWM,0.059464,0.0614,0.0036,-0.0179,-0.0110,-0.0175,0.0,0.059464
1,2015-03-31,IWM,0.017700,-0.0109,0.0308,-0.0038,0.0007,-0.0062,0.0,0.017700
2,2015-04-30,IWM,-0.025649,0.0060,-0.0301,0.0180,0.0005,-0.0062,0.0,-0.025649
3,2015-05-31,IWM,0.022364,0.0138,0.0082,-0.0111,-0.0176,-0.0083,0.0,0.022364
4,2015-06-30,IWM,0.007829,-0.0154,0.0290,-0.0082,0.0035,-0.0154,0.0,0.007829


In [5]:
ticker = "IWM"

data = df[df["ticker"] == ticker].copy()
data = data.sort_values("date")

y = data["excess_return"]
X = data[["mkt_rf", "smb", "hml", "rmw", "cma"]]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          excess_return   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.990
Method:                 Least Squares   F-statistic:                     2524.
Date:                Mon, 04 May 2026   Prob (F-statistic):          3.09e-125
Time:                        14:31:40   Log-Likelihood:                 496.41
No. Observations:                 133   AIC:                            -980.8
Df Residuals:                     127   BIC:                            -963.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0010      0.001     -1.805      0.0

In [6]:
regression_summary, df_predictions = kit.run_factor_regression(df)

In [7]:
regression_summary.sort_values("ticker")

,ticker,alpha,alpha_tstat,alpha_pvalue,beta_market,beta_market_tstat,beta_market_pvalue,beta_smb,beta_smb_tstat,beta_smb_pvalue,...,beta_hml_pvalue,beta_rmw,beta_rmw_tstat,beta_rmw_pvalue,beta_cma,beta_cma_tstat,beta_cma_pvalue,r_squared,adj_r_squared,n_obs
0,IWM,-0.000964,-1.805439,0.073375,1.000568,78.716227,1.217218e-109,0.816066,37.292463,2.841692e-70,...,4.671699e-07,-0.115385,-4.234658,4.357474e-05,-0.034876,-1.190843,0.235936,0.990035,0.989643,133
1,MTUM,0.000737,0.417910,0.676719,0.976609,23.252556,1.335359e-47,-0.220917,-3.055318,2.740522e-03,...,1.168512e-02,-0.179279,-1.991276,4.859651e-02,0.174783,1.806174,0.073259,0.824980,0.818089,133
2,QQQ,0.002426,1.960691,0.052104,1.099168,37.316790,2.633869e-70,-0.182041,-3.589942,4.710140e-04,...,1.996041e-09,-0.009181,-0.145400,8.846257e-01,-0.141128,-2.079517,0.039581,0.933993,0.931394,133
3,QUAL,-0.000817,-1.142107,0.255558,0.975596,57.268912,1.397681e-92,-0.078069,-2.661988,8.772456e-03,...,2.930758e-01,0.209962,5.749641,6.301423e-08,0.001249,0.031822,0.974664,0.968292,0.967043,133
4,SPY,-0.000187,-0.800338,0.425010,0.981940,176.262450,1.166262e-153,-0.106301,-11.083844,2.180925e-20,...,5.800473e-02,0.069996,5.861334,3.721506e-08,0.023812,1.855155,0.065893,0.996467,0.996328,133
5,USMV,-0.000828,-0.575192,0.566179,0.710765,20.731696,1.392789e-42,-0.121501,-2.058578,4.157951e-02,...,6.788350e-01,0.269796,3.671092,3.543340e-04,0.174232,2.205706,0.029204,0.794793,0.786713,133
6,VLUE,-0.001258,-0.946205,0.345841,0.985871,31.149074,2.553953e-61,0.143373,2.631298,9.559385e-03,...,2.825718e-12,0.033927,0.500056,6.179011e-01,0.083584,1.146187,0.253873,0.916592,0.913308,133
7,VTV,-0.000431,-0.470382,0.638889,0.873500,40.023882,7.218647e-74,-0.018879,-0.502474,6.162046e-01,...,1.462201e-14,0.100198,2.141743,3.412337e-02,0.183840,3.656001,0.000374,0.942589,0.940329,133
8,VUG,0.000337,0.418948,0.675962,1.092104,56.954953,2.737392e-92,-0.169359,-5.130427,1.053527e-06,...,8.915963e-14,-0.007101,-0.172752,8.631217e-01,-0.158633,-3.590626,0.000470,0.970257,0.969086,133


In [11]:
df_predictions = df_predictions.sort_values(["ticker", "date"]).copy()

df_predictions["cumulative_actual_return"] = (
    df_predictions
    .groupby("ticker")["return"]
    .transform(lambda x: (1 + x).cumprod() - 1)
)

df_predictions["cumulative_predicted_return"] = (
    df_predictions
    .groupby("ticker")["predicted_return"]
    .transform(lambda x: (1 + x).cumprod() - 1)
)

df_predictions["cumulative_residual"] = (
    df_predictions
    .groupby("ticker")["residual"]
    .transform(lambda x: (1 + x).cumprod() - 1)
)

In [12]:
df_predictions[df_predictions["ticker"] == "QQQ"][
    [
        "date",
        "ticker",
        "return",
        "predicted_return",
        "residual",
        "cumulative_actual_return",
        "cumulative_predicted_return",
    ]
].head()

,date,ticker,return,predicted_return,residual,cumulative_actual_return,cumulative_predicted_return
266,2015-02-28,QQQ,0.072206,0.077170,-0.004964,0.072206,0.077170
267,2015-03-31,QQQ,-0.023590,-0.013160,-0.010430,0.046913,0.062995
268,2015-04-30,QQQ,0.019223,0.010002,0.009221,0.067038,0.073627
269,2015-05-31,QQQ,0.022485,0.020746,0.001739,0.091030,0.095900
270,2015-06-30,QQQ,-0.024841,-0.015193,-0.009648,0.063927,0.079250


In [13]:
regression_summary.to_csv(
    PROCESSED_DIR / "factor_regression_summary.csv",
    index=False
)

df_predictions.to_csv(
    PROCESSED_DIR / "etf_ff5_with_predictions.csv",
    index=False
)